# Exploratory Data Analysis

Import packages.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd

from tqdm import tqdm
from utils.common import DATASETS, DATASET_TO_NUM_INSTANCES
from utils.model import load_eval_results
from utils.path import resolve_dataset_path, resolve_eval_results_dir

## Datasets

View the datasets we're interested in:
- [DS-1000](https://huggingface.co/datasets/xlangai/DS-1000)
- [MATH](https://huggingface.co/datasets/EleutherAI/hendrycks_math)
- [MMLU](https://huggingface.co/datasets/cais/mmlu) 

**Note:** we're not interested in  Chatbot-Arena/Chatbot-Arena_NEW and WildChat10K beacuse they evaluate preferences between only two models on each instance.

In [2]:
DATASETS

['DS-1000', 'MATH', 'MMLU']

Load each dataset.

In [3]:
dataset_to_df = {}

kwargs = {
    "desc": "Loading datasets",
    "total": len(DATASETS),
    "unit": "dataset",
}

for dataset in tqdm(DATASETS, **kwargs):
    file_path = resolve_dataset_path(dataset)
    dataset_to_df[dataset] = pd.read_json(file_path)

Loading datasets: 100%|██████████| 3/3 [00:00<00:00, 26.16dataset/s]


Count the number of instances in each dataset.

In [4]:
for dataset, df in dataset_to_df.items():
    print(f"{dataset}: {len(df)} instances")

    num_instances = DATASET_TO_NUM_INSTANCES[dataset]
    assert (
        len(df) == num_instances
    ), f"{dataset} has {len(df)} instances, but {num_instances} were expected"

DS-1000: 1000 instances
MATH: 5000 instances
MMLU: 14042 instances


## Evaluation Results

For each dataset, combine all of its evaluation results into a single CSV file.

In [7]:
dataset_to_eval_results = {}

kwargs = {
    "desc": "Loading evaluation results",
    "total": len(DATASETS),
    "unit": "dataset",
}

for dataset in tqdm(DATASETS, **kwargs):
    eval_results_dir = resolve_eval_results_dir(dataset)
    model_results = {}

    for model_dir in eval_results_dir.iterdir():
        if not model_dir.is_dir():
            continue
        results_file = model_dir / "results.json"
        with open(results_file) as f:
            model_results[model_dir.name] = json.load(f)

    df = pd.DataFrame(model_results)
    dataset_to_eval_results[dataset] = df
    df.to_csv(eval_results_dir / "all.csv", index=False)

Loading evaluation results: 100%|██████████| 3/3 [00:00<00:00, 58.60dataset/s]


View each dataset's evaluation results.

In [8]:
for dataset, df in dataset_to_eval_results.items():
    print(f"{dataset}: {len(df)} instances")
    display(df.head())

DS-1000: 1000 instances


,deepseek-coder-6.7b-base,gpt-3.5-turbo-0613,gpt-4o-2024-08-06
0,1,0,0
1,0,1,1
2,1,0,1
3,1,1,1
4,0,1,0


MATH: 5000 instances


,dart-math-llama3-8b-uniform,deepseek-math-7b-instruct,gpt-4o-mini-2024-07-18,Llama-3-8B-Instruct,Llama-3.1-8B-Instruct
0,1,1,1,1,1
1,1,1,1,0,1
2,0,1,1,1,0
3,1,0,1,0,1
4,1,1,1,1,1


MMLU: 14042 instances


,claude-3.5-haiku,gpt-3.5-turbo,gpt-4o-mini-2024-07-18,Llama-3.1-70B-Instruct,Llama-3.1-8B-Instruct,Llama-3.1-Tulu-3-70B,Llama-3.1-Tulu-3-8B,Qwen2.5-72B-Instruct,Qwen2.5-7B-Instruct
0,1,0,1,1,0,1,0,1,0
1,0,0,0,1,0,0,0,1,0
2,1,1,0,1,0,0,0,0,1
3,0,0,0,0,0,1,0,1,1
4,1,0,1,1,1,1,0,1,1
